# Style & Content Conditioned Language Modeling

Prototype CVAE model for transferring modern text to Shakespearean style.

Goal:
We want a generative model $p_{\theta}(x \mid c, s)$ which generates Shakespearean text samples $\bar{x} \sim p_{\theta}(x \mid c, s)$. 
- $c$ is the **content** representation derived from Modern English text.
- $s$ is the **style** representation derived from the Shakespearean target text via a VAE latent space.

At inference time, we provide a Modern translation, sample or extract a Shakespearean style vector, and generate the stylized equivalent.

In [5]:
## Paired Dataset & Dynamic Splitting
## Loads parallel sentences and splits them into train, val, and test sets

import torch
from pathlib import Path
from torch.utils.data import Dataset, random_split

DATA_DIR = Path("../../../shakespeare_data/align/plays/model_16plays")

class PairedShakespeareDataset(Dataset):
    """Parallel sentences from modern and original Shakespeare files."""

    def __init__(self, modern_file, shake_file):
        self.modern_lines = []
        self.shake_lines = []
        
        with open(modern_file, "r", encoding="utf-8") as fm, \
             open(shake_file, "r", encoding="utf-8") as fs:
            for mod, shk in zip(fm, fs):
                mod, shk = mod.strip(), shk.strip()
                if mod and shk:
                    self.modern_lines.append(mod)
                    self.shake_lines.append(shk)
                    
        assert len(self.modern_lines) == len(self.shake_lines), "Mismatched parallel data"

    def __len__(self):
        return len(self.modern_lines)

    def __getitem__(self, idx):
        return self.modern_lines[idx], self.shake_lines[idx]

# Load the full dataset
full_dataset = PairedShakespeareDataset(DATA_DIR / "train_plays.modern", DATA_DIR / "train_plays.original")

# Define split proportions (e.g., 80% train, 10% validate, 10% test)
total_size = len(full_dataset)
train_size = int(0.8 * total_size)
val_size = int(0.1 * total_size)
test_size = total_size - train_size - val_size

# Split the dataset using PyTorch's random_split
train, validate, test = random_split(
    full_dataset, 
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42) # Set a seed for reproducibility
)

print(f"Total pairs: {total_size}")
print(f"Train: {len(train)} | Val: {len(validate)} | Test: {len(test)}")

Total pairs: 19617
Train: 15693 | Val: 1961 | Test: 1963


In [6]:
## Shared Tokenizer
## We build the vocabulary from both modern and Shakespearean text 
## to ensure a shared embedding space.

import torch
from collections import Counter

MAX_VOCAB = None   
MAX_LENGTH = 64    

PAD, BOS, EOS, UNK = "<pad>", "<bos>", "<eos>", "<unk>"

class Vocab:
    def __init__(self, dataset, max_vocab=MAX_VOCAB):
        # Count words from both modern and shake lines
        counts = Counter(w for mod, shk in dataset for s in (mod, shk) for w in s.split())
        self.itos = [PAD, BOS, EOS, UNK] + [w for w, _ in counts.most_common(max_vocab)]
        self.stoi = {w: i for i, w in enumerate(self.itos)}
        self.pad_id, self.bos_id, self.eos_id, self.unk_id = 0, 1, 2, 3

    def __len__(self):
        return len(self.itos)

    def encode(self, sentence, max_length=MAX_LENGTH):
        ids = [self.stoi.get(w, self.unk_id) for w in sentence.split()]
        return [self.bos_id] + ids[: max_length - 2] + [self.eos_id]

    def decode(self, ids, strip_special=True):
        words = []
        for i in ids:
            i = int(i)
            if i == self.eos_id:
                break
            if strip_special and i in (self.pad_id, self.bos_id):
                continue
            words.append(self.itos[i])
        return " ".join(words)

vocab = Vocab(train)

def collate_fn(batch):
    """batch: list of (modern, shake) tuples -> padded tensors."""
    mod_seqs = [vocab.encode(mod) for mod, _ in batch]
    shk_seqs = [vocab.encode(shk) for _, shk in batch]
    
    def pad_seqs(seqs):
        T = max(len(s) for s in seqs)
        ids = torch.full((len(seqs), T), vocab.pad_id, dtype=torch.long)
        mask = torch.zeros(len(seqs), T, dtype=torch.long)
        for i, s in enumerate(seqs):
            ids[i, : len(s)] = torch.tensor(s)
            mask[i, : len(s)] = 1
        return ids, mask
        
    mod_ids, mod_mask = pad_seqs(mod_seqs)
    shk_ids, shk_mask = pad_seqs(shk_seqs)
    
    return {
        "modern_ids": mod_ids, "modern_mask": mod_mask,
        "shake_ids": shk_ids, "shake_mask": shk_mask
    }

print(f"Shared vocab size: {len(vocab)}")

Shared vocab size: 28403


In [7]:
## Dataloaders & Config
from torch.utils.data import DataLoader
import torch.nn as nn
import torch.nn.functional as F

BATCH_SIZE = 32

train_loader = DataLoader(train, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn, drop_last=True)
validate_loader = DataLoader(validate, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(test, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

VOCAB_SIZE = len(vocab)
PAD_ID, BOS_ID, EOS_ID, UNK_ID = vocab.pad_id, vocab.bos_id, vocab.eos_id, vocab.unk_id

D_MODEL = 128
N_HEAD = 4
N_LAYERS = 2
D_FF = 256
LATENT_DIM = 32
DROPOUT = 0.1
WORD_DROPOUT = 0.25

In [8]:
## Dual-Encoder CVAE Architecture
## Content Encoder processes Modern text.
## Style Encoder processes Shakespeare text to form latent z.
## Decoder generates Shakespeare using both Content (memory) and Style (z).

class StyleContentVAE(nn.Module):
    def __init__(self, vocab_size=VOCAB_SIZE, d_model=D_MODEL, nhead=N_HEAD,
                 num_layers=N_LAYERS, dim_feedforward=D_FF, latent_dim=LATENT_DIM,
                 dropout=DROPOUT, max_len=MAX_LENGTH, pad_id=PAD_ID, bos_id=BOS_ID, 
                 eos_id=EOS_ID, unk_id=UNK_ID, word_dropout=WORD_DROPOUT):
        super().__init__()
        self.latent_dim = latent_dim
        self.word_dropout = word_dropout
        self.pad_id, self.bos_id, self.eos_id, self.unk_id = pad_id, bos_id, eos_id, unk_id

        self.embed = nn.Embedding(vocab_size, d_model, padding_idx=pad_id)
        self.pos = nn.Embedding(max_len, d_model)
        self.drop = nn.Dropout(dropout)

        enc_layer = nn.TransformerEncoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, norm_first=True)
        self.content_encoder = nn.TransformerEncoder(enc_layer, num_layers, norm=nn.LayerNorm(d_model))
        self.style_encoder = nn.TransformerEncoder(enc_layer, num_layers, norm=nn.LayerNorm(d_model))

        self.to_mu = nn.Linear(d_model, latent_dim)
        self.to_logvar = nn.Linear(d_model, latent_dim)
        self.from_z = nn.Linear(latent_dim, d_model) 

        dec_layer = nn.TransformerDecoderLayer(d_model, nhead, dim_feedforward, dropout, batch_first=True, norm_first=True)
        self.decoder = nn.TransformerDecoder(dec_layer, num_layers, norm=nn.LayerNorm(d_model))

        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.embed.weight

        nn.init.normal_(self.embed.weight, std=0.02)
        nn.init.normal_(self.pos.weight, std=0.02)
        with torch.no_grad():
            self.embed.weight[pad_id].zero_()

    def _embed(self, ids):
        pos = torch.arange(ids.size(1), device=ids.device)
        return self.drop(self.embed(ids) + self.pos(pos))

    def encode_style(self, ids, mask):
        h = self.style_encoder(self._embed(ids), src_key_padding_mask=(mask == 0))
        m = mask.unsqueeze(-1).to(h.dtype)
        pooled = (h * m).sum(1) / m.sum(1).clamp(min=1.0)
        return self.to_mu(pooled), self.to_logvar(pooled)

    def encode_content(self, ids, mask):
        return self.content_encoder(self._embed(ids), src_key_padding_mask=(mask == 0))

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        return mu + std * torch.randn_like(std)

    def decode(self, z, content_memory, content_mask, input_ids, attention_mask=None):
        # Inject style z by prepending it to the content memory sequence
        z_mem = self.from_z(z).unsqueeze(1)                   
        full_memory = torch.cat([z_mem, content_memory], dim=1) 

        # Update cross-attention mask to include the z token
        if content_mask is not None:
            z_mask = torch.ones(content_mask.size(0), 1, dtype=content_mask.dtype, device=content_mask.device)
            full_mask = torch.cat([z_mask, content_mask], dim=1)
        else:
            full_mask = None

        T = input_ids.size(1)
        causal = torch.ones(T, T, dtype=torch.bool, device=input_ids.device).triu(1)
        h = self.decoder(
            self._embed(input_ids), full_memory,
            tgt_mask=causal, tgt_is_causal=True,
            tgt_key_padding_mask=None if attention_mask is None else (attention_mask == 0),
            memory_key_padding_mask=None if full_mask is None else (full_mask == 0)
        )
        return self.lm_head(h)

    def _word_dropout(self, ids):
        if not self.training or self.word_dropout <= 0: return ids
        drop = torch.rand(ids.shape, device=ids.device) < self.word_dropout
        drop &= ids != self.bos_id
        drop &= ids != self.pad_id
        return ids.masked_fill(drop, self.unk_id)

    def forward(self, modern_ids, modern_mask, shake_ids, shake_mask):
        mu, logvar = self.encode_style(shake_ids, shake_mask)
        z = self.reparameterize(mu, logvar)
        
        content_memory = self.encode_content(modern_ids, modern_mask)
        
        dec_in = self._word_dropout(shake_ids[:, :-1])
        logits = self.decode(z, content_memory, modern_mask, dec_in, shake_mask[:, :-1])
        
        targets = shake_ids[:, 1:].clone()
        targets[shake_mask[:, 1:] == 0] = -100
        
        return {"logits": logits, "targets": targets, "mu": mu, "logvar": logvar, "z": z}

    @torch.no_grad()
    def generate(self, modern_ids, modern_mask, z=None, max_new_tokens=40, temperature=1.0, top_k=None):
        self.eval()
        device = modern_ids.device
        
        if z is None:
            z = torch.randn(modern_ids.size(0), self.latent_dim, device=device)
            
        content_memory = self.encode_content(modern_ids, modern_mask)
        
        ids = torch.full((modern_ids.size(0), 1), self.bos_id, dtype=torch.long, device=device)
        done = torch.zeros(modern_ids.size(0), dtype=torch.bool, device=device)
        
        for _ in range(max_new_tokens):
            logits = self.decode(z, content_memory, modern_mask, ids)[:, -1] / temperature
            logits[:, self.unk_id] = float("-inf")
            if top_k is not None:
                kth = logits.topk(min(top_k, logits.size(-1)), dim=-1).values[:, -1:]
                logits = logits.masked_fill(logits < kth, float("-inf"))
            
            nxt = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            nxt[done] = self.pad_id
            ids = torch.cat([ids, nxt], dim=1)
            done |= nxt.squeeze(1) == self.eos_id
            if done.all():
                break
        return ids

In [9]:
## The ELBO Loss remains exactly the same as the unconditional VAE.
## We reuse elbo_loss and beta_schedule from the previous code.

def elbo_loss(out, beta=1.0, free_bits=0.0):
    logits, targets, mu, logvar = out["logits"], out["targets"], out["mu"], out["logvar"]
    batch_size = targets.size(0)

    recon = F.cross_entropy(
        logits.reshape(-1, logits.size(-1)), targets.reshape(-1), ignore_index=-100, reduction="sum"
    ) / batch_size

    kl_dim = -0.5 * (1 + logvar - mu.pow(2) - logvar.exp())
    kl_dim = kl_dim.mean(0)
    if free_bits > 0:
        kl_dim = kl_dim.clamp(min=free_bits)
    kl = kl_dim.sum()

    n_tokens = (targets != -100).sum().clamp(min=1)
    return {
        "loss": recon + beta * kl, "recon": recon.detach(),
        "kl": kl.detach(), "ppl": (recon * batch_size / n_tokens).exp().detach()
    }

def beta_schedule(step, warmup_steps, max_beta=1.0):
    return max_beta * min(1.0, step / max(1, warmup_steps))

def train_cvae(model, train_loader, epochs=10, lr=3e-4, warmup_steps=1000, free_bits=0.2):
    model.to(DEVICE)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    step = 0

    for epoch in range(1, epochs + 1):
        model.train()
        for batch in train_loader:
            mod_ids, mod_mask = batch["modern_ids"].to(DEVICE), batch["modern_mask"].to(DEVICE)
            shk_ids, shk_mask = batch["shake_ids"].to(DEVICE), batch["shake_mask"].to(DEVICE)

            beta = beta_schedule(step, warmup_steps)
            stats = elbo_loss(model(mod_ids, mod_mask, shk_ids, shk_mask), beta=beta, free_bits=free_bits)

            opt.zero_grad()
            stats["loss"].backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            step += 1

            if step % 500 == 0:
                print(f"Epoch {epoch} | Step {step} | Loss: {stats['loss']:.2f} | KL: {stats['kl']:.2f} | Recon: {stats['recon']:.2f}")

In [15]:
## Model Initialization and Training
## Run this cell to train the StyleContentVAE model.

import torch

# Set seed for reproducibility during initialization
torch.manual_seed(42)

# Initialize the dual-encoder model
model = StyleContentVAE().to(DEVICE)
print(f"Model initialized with {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters on {DEVICE}")

# Define training hyperparameters
EPOCHS = 50  # Adjust as needed based on time and hardware
LEARNING_RATE = 3e-4

print(f"Starting training for {EPOCHS} epochs...")

# Run the training loop (using the train_cvae function defined in previous cells)
train_cvae(
    model=model, 
    train_loader=train_loader, 
    epochs=EPOCHS, 
    lr=LEARNING_RATE, 
    warmup_steps=1000, 
    free_bits=0.2
)

# Save the trained model checkpoint
torch.save({
    "model_state_dict": model.state_dict(),
    "vocab_itos": vocab.itos,
}, "style_content_vae.pt")

print("Training complete. Saved checkpoint to style_content_vae.pt")

/scratch/8001198.1.cds-gpu/ipykernel_630819/1186995832.py:21: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.content_encoder = nn.TransformerEncoder(enc_layer, num_layers, norm=nn.LayerNorm(d_model))
/scratch/8001198.1.cds-gpu/ipykernel_630819/1186995832.py:22: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.style_encoder = nn.TransformerEncoder(enc_layer, num_layers, norm=nn.LayerNorm(d_model))


Model initialized with 4.58M parameters on cuda
Starting training for 50 epochs...
Epoch 2 | Step 500 | Loss: 77.30 | KL: 6.40 | Recon: 74.10
Epoch 3 | Step 1000 | Loss: 76.66 | KL: 6.40 | Recon: 70.26
Epoch 4 | Step 1500 | Loss: 65.85 | KL: 6.40 | Recon: 59.45
Epoch 5 | Step 2000 | Loss: 58.31 | KL: 6.47 | Recon: 51.84
Epoch 6 | Step 2500 | Loss: 71.31 | KL: 6.48 | Recon: 64.83
Epoch 7 | Step 3000 | Loss: 62.51 | KL: 6.40 | Recon: 56.11
Epoch 8 | Step 3500 | Loss: 55.53 | KL: 6.42 | Recon: 49.11
Epoch 9 | Step 4000 | Loss: 75.14 | KL: 6.43 | Recon: 68.71
Epoch 10 | Step 4500 | Loss: 51.40 | KL: 6.40 | Recon: 45.00
Epoch 11 | Step 5000 | Loss: 49.36 | KL: 6.40 | Recon: 42.96
Epoch 12 | Step 5500 | Loss: 51.59 | KL: 6.48 | Recon: 45.12
Epoch 13 | Step 6000 | Loss: 35.90 | KL: 6.40 | Recon: 29.50
Epoch 14 | Step 6500 | Loss: 54.07 | KL: 6.44 | Recon: 47.64
Epoch 15 | Step 7000 | Loss: 53.61 | KL: 6.41 | Recon: 47.21
Epoch 16 | Step 7500 | Loss: 39.85 | KL: 6.45 | Recon: 33.39
Epoch 17 | 

In [16]:
## Evaluation & Generation
## Set NUM_SAMPLES to any number, and run this cell repeatedly for different examples.

import random

# Number of samples you want to inspect
NUM_SAMPLES = 10  # change this to any number you want

# Set up an evaluation DataLoader with shuffle=True so each run yields different samples
eval_loader = DataLoader(
    validate, 
    batch_size=NUM_SAMPLES, 
    shuffle=True, 
    collate_fn=collate_fn
)

model.eval()
print(f"\n--- Model Evaluation ({NUM_SAMPLES} Random Validation Samples) ---")

batch = next(iter(eval_loader))

mod_ids = batch["modern_ids"].to(DEVICE)
mod_mask = batch["modern_mask"].to(DEVICE)
shk_ids = batch["shake_ids"].to(DEVICE)
shk_mask = batch["shake_mask"].to(DEVICE)

with torch.no_grad():
    # Style: encode mu from target, or pass z=None to sample randomly from N(0, I)
    mu, _ = model.encode_style(shk_ids, shk_mask)
    
    # Generate Shakespeare conditioned on modern content and style mu
    generated_ids = model.generate(mod_ids, mod_mask, z=mu, top_k=40)

for i in range(len(mod_ids)):
    modern_text = vocab.decode(mod_ids[i])
    actual_shake = vocab.decode(shk_ids[i])
    gen_shake = vocab.decode(generated_ids[i])
    
    print(f"[{i+1}/{NUM_SAMPLES}]")
    print(f"Modern Input : {modern_text}")
    print(f"Actual Target: {actual_shake}")
    print(f"Generated    : {gen_shake}")
    print("-" * 60)


--- Model Evaluation (10 Random Validation Samples) ---
[1/10]
Modern Input : Good morning to both of you.
Actual Target: Good morrow, both.
Generated    : Good morrow to you.
------------------------------------------------------------
[2/10]
Modern Input : I swear to you by <unk> own pretty hand that I am that unfortunate man.
Actual Target: I swear to thee, youth, by the white hand of Rosalind, I am that he, that unfortunate he.
Generated    : I durst, my face in hand, I’ll return to thy own soul and fortune, Thou know’st, most unfortunate man.
------------------------------------------------------------
[3/10]
Modern Input : <unk> in the bill, sir, error in the <unk>
Actual Target: <unk> i' the bill, sir, error i' the <unk>
Generated    : I' faith, thou call’st on the invisible of the another.
------------------------------------------------------------
[4/10]
Modern Input : My old brain is troubled.
Actual Target: My old brain is troubled.
Generated    : Madam, grow most unquietl

In [2]:
!pwd
!df -h .

/projectnb/scottml/gamato23/marthabot/code/notebooks


Filesystem                  Size  Used Avail Use% Mounted on
scc-fs5vp:/gpfs4/projectnb  550G  317G  234G  58% /projectnb
